# 01 — Data Download

Downloads raw Amazon product metadata from `McAuley-Lab/Amazon-Reviews-2023` on HuggingFace.

**What it does:**
- Fetches fields: `parent_asin`, `title`, `description`, `features`, `main_category`
- Saves one `.parquet` file per Amazon category to `data/category_cache/`
- Resumable: already-downloaded categories are skipped

**Does NOT** clean, map, or deduplicate — that is done in `02_data_cleaning.ipynb`.

**Setup:**
```bash
conda activate prod_classifier
jupyter notebook notebooks/01_data_download.ipynb
```

In [ ]:
import os
import logging
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from datasets import load_dataset

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s  %(levelname)s  %(message)s',
    datefmt='%H:%M:%S'
)
log = logging.getLogger(__name__)

PROJ      = '/home/tayana-gpu/ML/project_01_product_classifier'
CACHE_DIR = f'{PROJ}/data/category_cache'
RAW_FIELDS = ['parent_asin', 'title', 'description', 'features', 'main_category']

os.makedirs(CACHE_DIR, exist_ok=True)
print(f'Cache dir: {CACHE_DIR}')

In [ ]:
# All 30 meaningful categories from McAuley-Lab/Amazon-Reviews-2023.
# Gift_Cards, Subscription_Boxes, Unknown are excluded — no valid product metadata.
CATEGORIES = [
    'raw_meta_All_Beauty',
    'raw_meta_Amazon_Fashion',
    'raw_meta_Appliances',
    'raw_meta_Arts_Crafts_and_Sewing',
    'raw_meta_Automotive',
    'raw_meta_Baby_Products',
    'raw_meta_Beauty_and_Personal_Care',
    'raw_meta_Books',
    'raw_meta_CDs_and_Vinyl',
    'raw_meta_Cell_Phones_and_Accessories',
    'raw_meta_Clothing_Shoes_and_Jewelry',
    'raw_meta_Digital_Music',
    'raw_meta_Electronics',
    'raw_meta_Grocery_and_Gourmet_Food',
    'raw_meta_Handmade_Products',
    'raw_meta_Health_and_Household',
    'raw_meta_Health_and_Personal_Care',
    'raw_meta_Home_and_Kitchen',
    'raw_meta_Industrial_and_Scientific',
    'raw_meta_Kindle_Store',
    'raw_meta_Magazine_Subscriptions',
    'raw_meta_Movies_and_TV',
    'raw_meta_Musical_Instruments',
    'raw_meta_Office_Products',
    'raw_meta_Patio_Lawn_and_Garden',
    'raw_meta_Pet_Supplies',
    'raw_meta_Software',
    'raw_meta_Sports_and_Outdoors',
    'raw_meta_Tools_and_Home_Improvement',
    'raw_meta_Toys_and_Games',
    'raw_meta_Video_Games',
]

print(f'Categories to download: {len(CATEGORIES)}')

In [ ]:
def download_category(config_name: str) -> None:
    out_path = f'{CACHE_DIR}/{config_name}.parquet'

    if os.path.exists(out_path):
        existing = pd.read_parquet(out_path, columns=['parent_asin'])
        log.info('SKIP  %s  (%d rows already cached)', config_name, len(existing))
        return

    log.info('START %s', config_name)

    try:
        dataset = load_dataset(
            'McAuley-Lab/Amazon-Reviews-2023',
            config_name,
            split='full',
            trust_remote_code=True,
        )
    except Exception as exc:
        log.error('FAIL  %s — %s', config_name, exc)
        return

    log.info('FETCH %s — %d rows', config_name, len(dataset))

    # Write in chunks so we never hold the full dataset in RAM.
    CHUNK = 200_000
    writer = None
    total  = 0
    try:
        for start in range(0, len(dataset), CHUNK):
            batch   = dataset[start: start + CHUNK]
            records = {f: batch.get(f) for f in RAW_FIELDS if f in batch}
            table   = pa.Table.from_pydict(records)
            if writer is None:
                writer = pq.ParquetWriter(out_path, table.schema, compression='snappy')
            writer.write_table(table)
            total += len(table)
        if writer:
            writer.close()
    except Exception:
        if writer:
            writer.close()
        if os.path.exists(out_path):
            os.remove(out_path)
        raise

    log.info('SAVE  %s — %d rows → %s', config_name, total, out_path)

In [ ]:
log.info('McAuley-Lab/Amazon-Reviews-2023 — raw download')
log.info('Categories: %d', len(CATEGORIES))
log.info('Cache dir:  %s', CACHE_DIR)

for config_name in CATEGORIES:
    download_category(config_name)

log.info('All downloads complete.')
log.info('Next step: run 02_data_cleaning.ipynb')